# Objective

Built a chatbot to call tools using yfinance library.

## Import Libraries

In [17]:
import yfinance as yf
import anthropic

from dotenv import load_dotenv

#### Input parameters ####
model = "claude-sonnet-4-20250514"
#########################

## Tool Functions

In [18]:
def get_price(stock_code: str) -> float:
    """
    Get stock data from Yahoo Finance.

    Args:
        stock_code (str): The stock code to fetch data for.

    Returns:
        float: The current stock price.
    """

    stk = yf.Ticker(stock_code)

    try:
        stk_price = stk.info['currentPrice']
    except:
        try:
            stk_price = stk.info['previousClose']
        except:
            print("Error!!!!!! Unable to get stk_price.")

    return stk_price

In [19]:
# Test
print(get_price("AAPL"))

269.05


In [20]:
def get_analyst_price_target(stock_code: str) -> float:
    """
    Get analyst price targets from Yahoo Finance.

    Args:
        stock_code (str): The stock code to fetch data for.

    Returns:
        float: The mean analyst price target.
    """
    stk = yf.Ticker(stock_code)

    try:
        price_target = stk.analyst_price_targets['mean']
    except:
        print("Error!!!!!! Unable to get analyst price targets.")

    return price_target

In [21]:
# Test
get_analyst_price_target("AAPL")

278.74805

## Tool Schema

Here are the schema of each tool which you will provide to the LLM.

In [22]:
tools = [
    {
        "name": "get_price",
        "description": "Get the current stock price for a given stock code.",
        "input_schema": {
            "type": "object",
            "properties": {
                "stock_code": {
                    "type": "string",
                    "description": "The stock code to fetch data for"
                }
            },
            "required": ["stock_code"]
        }
    },
    {
        "name": "get_analyst_price_target",
        "description": "Get the mean analyst price target for a given stock code.",
        "input_schema": {
            "type": "object",
            "properties": {
                "stock_code": {
                    "type": "string",
                    "description": "The stock code to fetch data for"
                }
            },
            "required": ["stock_code"]
        }
    }
]

## Tool Mapping

This code handles tool mapping and execution.

In [23]:
mapping_tool_function = {
    "get_price": get_price,
    "get_analyst_price_target": get_analyst_price_target
}

def execute_tool(tool_name, tool_args):
    
    result = mapping_tool_function[tool_name](**tool_args)

    if result is None:
        result = "The operation completed but didn't return any results."
        
    elif isinstance(result, list):
        result = ', '.join(result)
        
    elif isinstance(result, dict):
        # Convert dictionaries to formatted JSON strings
        result = json.dumps(result, indent=2)
    
    else:
        # For any other type, convert using str()
        result = str(result)
    return result

## Chatbot Code

The chatbot handles the user's queries one by one, but it does not persist memory across the queries.

In [24]:
load_dotenv() 
client = anthropic.Anthropic()

### Query Processing

In [ ]:
def process_query(query):
    
    messages = [{'role': 'user', 'content': query}]
    
    response = client.messages.create(max_tokens=2024,
                                      model=model, 
                                      tools=tools,
                                      messages=messages)
    
    process_query = True
    while process_query:
        # Collect ALL content from assistant's response first
        assistant_content = []
        tool_results = []
        
        for content in response.content:
            if content.type == 'text':
                print(content.text)
                assistant_content.append(content)
                
            elif content.type == 'tool_use':
                assistant_content.append(content)
                
                tool_id = content.id
                tool_args = content.input
                tool_name = content.name
                print(f"Calling tool {tool_name} with args {tool_args}")
                
                result = execute_tool(tool_name, tool_args)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": tool_id,
                    "content": result
                })
        
        # Now append the assistant message ONCE with all content
        messages.append({'role': 'assistant', 'content': assistant_content})
        
        # If there were tool uses, add tool results and continue
        if tool_results:
            messages.append({"role": "user", "content": tool_results})
            
            response = client.messages.create(max_tokens=2024,
                                              model=model, 
                                              tools=tools,
                                              messages=messages)
            
            # Check if we're done
            if len(response.content) == 1 and response.content[0].type == "text":
                print(response.content[0].text)
                process_query = False
        else:
            # No tool uses, we're done
            process_query = False

In [29]:
process_query("What is the current stock price of AAPL and its mean analyst price target?")

response.content =  [TextBlock(citations=None, text="I'll get the current stock price and mean analyst price target for AAPL (Apple Inc.) for you.", type='text'), ToolUseBlock(id='toolu_01X4PCm5uonSJUpWRkVkckWs', input={'stock_code': 'AAPL'}, name='get_price', type='tool_use'), ToolUseBlock(id='toolu_01EKzeD2X4AMdQ4oExAD1w1x', input={'stock_code': 'AAPL'}, name='get_analyst_price_target', type='tool_use')]
I'll get the current stock price and mean analyst price target for AAPL (Apple Inc.) for you.
Calling tool get_price with args {'stock_code': 'AAPL'}
Calling tool get_analyst_price_target with args {'stock_code': 'AAPL'}
Here's the information for AAPL:

- **Current Stock Price**: $269.05
- **Mean Analyst Price Target**: $278.75

The analyst price target suggests that analysts expect AAPL to appreciate by approximately $9.70 per share (or about 3.6%) from its current price level.


## Resources

[Guide on how to implement tool use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/overview#how-to-implement-tool-use)